# 🩺 RSNA Knee Abnormality Detection — Starter Notebook

**What this notebook is.** A small, complete, *runnable* baseline: it reads the competition
metadata, turns the free-text radiology reports into weak training labels, builds a compact
image tensor per study from DICOM, trains one CNN, validates on the 58 fully-labelled studies,
and writes `submission.csv`.

It is deliberately simple. The companion notebook in this folder (`rsna-baseline.ipynb`) is a
copy of a 0.936 solution — a four-branch ensemble of pre-trained DINOv2 / DINOv3 / RadImageNet /
CoAtNet models. That notebook is *inference only*; you cannot learn from it how the pipeline is
built. This one shows you the whole chain end to end, at a size you can actually iterate on.

---

## The competition in one page

**Task.** Given a knee MRI *study*, predict the probability of **12 findings**:

| | Finding | What it is |
|---|---|---|
| 1 | `ACL` | Anterior cruciate ligament tear |
| 2 | `MCL` | Medial collateral ligament injury |
| 3 | `Medial Meniscus` | Medial meniscus tear |
| 4 | `Lateral Meniscus` | Lateral meniscus tear |
| 5 | `Medial OA` | Osteoarthritis, medial tibiofemoral compartment |
| 6 | `Lateral OA` | Osteoarthritis, lateral tibiofemoral compartment |
| 7 | `PF OA` | Osteoarthritis, patellofemoral compartment |
| 8 | `Effusion` | Joint effusion (fluid in the joint) |
| 9 | `Synovitis` | Inflamed / thickened synovium |
| 10 | `Baker's` | Baker's (popliteal) cyst |
| 11 | `Contusion` | Bone marrow contusion / bruise |
| 12 | `Fracture` | Fracture |

This is **multi-label**, not multi-class: a knee can have several findings at once, and most
have none of some and several of others.

**Metric.** Macro-averaged ROC AUC — the AUC is computed per finding and then averaged with
equal weight. Consequences worth internalising:

* Only the *ranking* of your predictions matters, not their calibration. A column of raw logits
  scores the same as perfectly calibrated probabilities.
* Every finding counts the same. `Fracture` (rare) is worth exactly as much as `Effusion`
  (common). Do not let the frequent labels dominate your loss and your attention.
* Because it is averaged over columns, gains on the *weakest* column are the cheapest gains
  available to you.

**Files.**

| File | What's in it |
|---|---|
| `train.csv` | 4,407 studies: `StudyInstanceUID`, `Report`, and the 12 label columns |
| `train_series.csv` | one row per series: plane, fluid-sensitivity, fat-suppression flags |
| `test.csv`, `test_series.csv` | the same, for the test studies |
| `train_series/<StudyUID>/<SeriesUID>/*.dcm` | the images |
| `sample_submission.csv` | `StudyInstanceUID` + 12 probability columns |

---

## ⚠️ The one thing that defines this competition

**Only 58 of the 4,407 training studies have labels.** The other 4,349 have `NaN` in every
label column — all you get for them is a free-text `Report` written by the reporting
radiologist, in whatever language that hospital works in (English, German, Spanish, Dutch,
Croatian, French …).

So the competition is really two problems stacked:

1. **A weak-supervision / NLP problem.** Turn 4,349 multilingual reports into usable labels.
   Everyone in the top of the leaderboard did some version of this — the 0.936 solution used an
   LLM to convert each report into 12 *soft* probabilities and trained on 4,349 studies. Its
   own notes report that expanding the corpus from 3,155 to 4,349 studies moved the gold-set
   score from 0.8923 to 0.9054, with the biggest jumps on the weakest labels
   (Lateral Meniscus +0.071, Fracture +0.057).
2. **A medical-imaging problem.** Turn a multi-series 3D study into a fixed-size tensor and
   train a model on it.

And a validation problem on top: with 58 labelled studies, *any* single number you compute has
an enormous error bar. Treat the gold set as a smoke test, not as a leaderboard.

This notebook does a rule-based version of step 1 (transparent, no LLM, no internet) and a
single small CNN for step 2.

## 1. Configuration

Everything you are likely to tune lives in this one cell. The defaults are sized to finish in a
couple of hours on a single T4; raise `MAX_TRAIN_STUDIES`, `N_TRIPLET` and `IMG` once the whole
chain runs for you.

In [ ]:
from __future__ import annotations

import glob
import json
import os
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------- competition constants
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)

# ----------------------------------------------------------------- what a study looks like to the model
# One "slot" = one MRI series we want. A knee study has ~5-6 series; we keep three, one per
# anatomical plane, preferring fluid-sensitive ones (T2/PD/STIR) because fluid, oedema and
# tears are far more conspicuous on them.
SLOTS = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]
N_SLOT = len(SLOTS)

N_TRIPLET = 4        # windows sampled per slot (a window = 3 adjacent slices -> 3 channels)
IMG = 192            # pixels per side after cropping and resizing
CROP_MM = 130.0      # physical field of view kept around the image centre, in millimetres
SLICE_BAND = (0.15, 0.85)   # ignore the first/last 15% of a stack: mostly air and soft tissue
K = N_SLOT * N_TRIPLET      # windows per study -> the model sees K images of 3 channels each

# ----------------------------------------------------------------- training
BACKBONE = "resnet18"        # try tf_efficientnet_b0, convnext_tiny, resnet34 ...
PRETRAINED = True            # needs Kaggle "Internet: on", or a timm-weights dataset attached
EPOCHS = 4
BATCH = 8
LR_HEAD = 3e-4
LR_BACKBONE = 1e-4
NUM_WORKERS = 2
SEED = 2026

# Cap the number of weakly-labelled studies used for training. Decoding DICOM is the slow part,
# so start small, confirm the pipeline works, then raise this towards 4349.
MAX_TRAIN_STUDIES = 600

# ----------------------------------------------------------------- paths
def find_root() -> Path:
    """Locate the competition data. Works on Kaggle (both mount layouts) and locally."""
    candidates = [
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("."),
    ]
    for c in candidates:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Could not find the competition data; set ROOT by hand.")


def find_csv(root: Path, stem: str) -> Path:
    """`train.csv` on Kaggle, but a local download may be named `train (10).csv`."""
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]


ROOT = find_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(exist_ok=True, parents=True)
CACHE = WORK / "cache"
CACHE.mkdir(exist_ok=True, parents=True)

np.random.seed(SEED)
print("data root :", ROOT.resolve())
print("work dir  :", WORK.resolve())
print("windows per study:", K, f"({N_SLOT} slots x {N_TRIPLET} triplets), {IMG}x{IMG} px")

## 2. Load the metadata and look at it

Before touching a single DICOM, get the shape of the problem into your head: how many studies,
how many are labelled, how the labels are balanced, and what series each study actually has.

In [ ]:
train = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test = pd.read_csv(find_csv(ROOT, "test"))
test_series = pd.read_csv(find_csv(ROOT, "test_series"))

for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

print(f"train studies      : {len(train)}")
print(f"test studies       : {len(test)}   <- tiny; the real test set appears only at rerun time")
print(f"train series rows  : {len(train_series)}")

# The 58 "gold" studies: the only ones with real, human-verified labels.
gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
weak = train.loc[~gold_mask].reset_index(drop=True)
print(f"\nfully labelled (gold)  : {len(gold)}")
print(f"report-only (weak)     : {len(weak)}")

print("\npositives among the 58 gold studies:")
print(gold[TARGETS].sum().astype(int).to_string())

print("\nseries per study:", train_series.groupby("StudyInstanceUID").size().describe()[["mean", "min", "max"]].to_dict())
print("\nplanes:\n", train_series["Anatomical_Plane"].value_counts().to_string())
print("\nfluid-sensitive flag:\n", train_series["Fluid_Sensitive"].value_counts().to_string())

In [ ]:
# What a report actually looks like. Note: multiple languages, free-form, and the sections
# ("Findings", "Conclusion") vary by hospital.
for i in [0, 1, 2]:
    print("-" * 100)
    print(train["Report"].iloc[i][:600])

## 3. Weak labels from the reports

This is where the competition is won or lost, so it is worth being explicit about what we are
doing and what its limits are.

**The idea.** A report that says *"Rotura de menisco interno"* is telling us
`Medial Meniscus = 1`. One that says *"Normal medial and lateral menisci"* is telling us
`Medial Meniscus = 0`. One that says nothing about the menisci tells us **nothing** — and that
is a third state we must keep, not silently collapse to 0.

So each `(study, finding)` pair gets one of three values: `1`, `0`, or `NaN` = unknown. The
training loss will simply **mask out the unknowns**. Turning "not mentioned" into a hard 0 is the
single most common way to poison a weak-label pipeline: radiologists routinely omit negatives.

**The method here** is deliberately transparent: multilingual keyword patterns plus a negation
window. For the ligaments and menisci we additionally require an abnormality word
(*tear / rupture / Riss / rotura / scheur …*) near the anatomy word, because "ACL" appearing in a
report is not evidence of an ACL tear. For the three osteoarthritis columns we look for an
OA-family word (*arthrosis / Knorpel / osteophyte / hrskavica …*) **co-occurring** with a
compartment word (*medial / lateral / patellofemoral*) within ~60 characters.

**Its limits, stated honestly.** Reports and images disagree. Among the 58 gold studies there are
cases labelled `Baker's = 1` whose report never mentions a cyst, and one whose report says
*"Baker-Zyste"* while the gold label is 0. The reports are a *noisy teacher*, not ground truth.
This is exactly why the strong solutions used an LLM to emit **soft** probabilities rather than
hard 0/1 — a soft label carries the model's uncertainty into the loss instead of asserting a
possibly-wrong 1.

In [ ]:
def normalise(text: str) -> str:
    """Lowercase, strip accents, collapse whitespace -> one comparable string across languages."""
    t = unicodedata.normalize("NFKD", str(text))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t.lower())


# Negation cues in the languages present in the corpus (en/de/es/nl/hr/fr).
NEG = (r"(?:no |not |without |absence of |negative for |intact |normal |unremarkable |ohne |"
       r"kein[e]?[nrms]? |unauff|sin |ausencia|geen |zonder |normale |normaal |bez |uredn|"
       r"nema |sans |pas de |absence)")

# Anatomy / finding cues.
PATTERNS = {
    "ACL": r"(acl|anterior cruciate|lca|vkb|ligamento cruzado anterior|voorste kruisband|"
           r"kruisband anterior|prednj[ei] krizn|kreuzband(?:ruptur)?\s*(?:vorder)?|vorderes kreuzband)",
    "MCL": r"(mcl|medial collateral|ligamento colateral medial|innenband|mediale[nr]? kollateralband|"
           r"mediale collaterale|medijalni kolateralni)",
    "Medial Meniscus": r"(medial meniscus|menisco (?:interno|medial)|innenmeniskus|mediale meniscus|"
                       r"medijalni menisk|meniscus medialis|meniscus internus)",
    "Lateral Meniscus": r"(lateral meniscus|menisco (?:externo|lateral)|aussenmeniskus|laterale meniscus|"
                        r"lateralni menisk|meniscus lateralis)",
    "Medial OA": None,   # handled by the compartment co-occurrence rule below
    "Lateral OA": None,  # handled by the compartment co-occurrence rule below
    "PF OA": None,       # handled by the compartment co-occurrence rule below
    "Effusion": r"(effusion|derrame|gelenkerguss|ergus[s]?|hydrops|izljev|epanchement|joint fluid|"
                r"gewrichtsvocht|vocht)",
    "Synovitis": r"(synovit\w*|sinovit\w*|synovialit\w*|sinovij\w*|"
                 r"synovial\w* (?:proliferation|thickening|verdikking|hypertroph\w*)|"
                 r"proliferacij\w* sinovij|pannus|synoviale? reizung)",
    "Baker's": r"(baker|popliteal cyst|quiste de baker|bakerzyste|baker-zyste|bakerova cist|kyste de baker)",
    "Contusion": r"(bone (?:marrow )?(?:contusion|bruise|oedema|edema)|contusion|knochenmarkod|kontuzij|"
                 r"botcontusie|edema oseo)",
    "Fracture": r"(fracture|fractur|fraktur|fisura osea|prijelom|breuk|fractuur|avulsion)",
}

# "ACL" alone is not a finding; we need a word saying something is wrong with it.
ABNORMAL = (r"(tear|rupt|riss|scheur|rotura|lesion|desgarr|lasion|laesion|degenerativ|signal|"
            r"tearing|ruptura|insuffizienz|discontinu|abnormal)")
NEEDS_ABNORMAL = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}

# Osteoarthritis is graded per compartment, so we need an OA word AND a compartment word.
OA_WORD = (r"(osteoarthrit\w*|arthros\w*|artros\w*|artroz\w*|gonarthros\w*|osteoartr\w*|chondral loss|"
           r"cartilage loss|knorpel\w*|chondropath\w*|kraakbeen\w*|hrskavic\w*|osteophyt\w*|osteofit\w*|"
           r"degenerative (?:change|veranderung)\w*|denudacij\w*)")
COMPARTMENT = {
    "Medial OA": r"(medial\w*|mediaal|medijaln\w*|intern[oa]|innen\w*|inner)",
    "Lateral OA": r"(lateral\w*|lateraal|lateraln\w*|extern[oa]|aussen\w*|outer)",
    "PF OA": r"(patell\w*|patel\w*|femoropatel\w*|retropatell\w*|trochlea\w*|trohlej\w*)",
}
WINDOW = 60  # characters either side, for the OA co-occurrence test


def _oa_label(t: str, key: str) -> float:
    pos = neg = 0
    for m in re.finditer(OA_WORD, t):
        s, e = m.span()
        ctx = t[max(0, s - WINDOW):e + WINDOW]
        if not re.search(COMPARTMENT[key], ctx):
            continue
        if key != "PF OA" and re.search(r"(?:patell|patel|trochlea|trohlej)", ctx):
            continue  # patellofemoral compartment -> that is PF OA, a different column
        if re.search(NEG + r"[^.]{0,25}$", t[max(0, s - 45):s]):
            neg += 1
        else:
            pos += 1
    return 1.0 if pos else (0.0 if neg else np.nan)


def label_report(text: str) -> dict:
    """Return {finding: 1.0 | 0.0 | nan} for one report. nan means 'the report does not say'."""
    t = normalise(text)
    out = {k: _oa_label(t, k) for k in COMPARTMENT}
    for key, pattern in PATTERNS.items():
        if pattern is None:
            continue
        pos = neg = 0
        for m in re.finditer(pattern, t):
            s, e = m.span()
            left, right = t[max(0, s - 45):s], t[e:e + 80]
            negated = (re.search(NEG + r"[^.]{0,25}$", left)
                       or re.search(r"^\W{0,4}(?:" + NEG + r"|ist intakt|intacto|intact)", right))
            if key in NEEDS_ABNORMAL:
                near_abnormal = re.search(ABNORMAL, right[:60]) or re.search(ABNORMAL + r"[^.]{0,30}$", left)
                if not near_abnormal:
                    # "ACL normal" / "menisci intact" -> a real negative; anything else -> unknown
                    if re.search(r"^\W{0,6}(?:" + NEG + r"|intact|normal)", right):
                        neg += 1
                    continue
            if negated:
                neg += 1
            else:
                pos += 1
        out[key] = 1.0 if pos else (0.0 if neg else np.nan)
    return out


weak_labels = pd.DataFrame([label_report(r) for r in train["Report"]])[TARGETS]
weak_labels.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)

coverage = weak_labels[TARGETS].notna().mean()
print("share of studies where the report lets us decide the label:")
print((coverage * 100).round(1).to_string())
print(f"\nstudies with at least one usable label: {weak_labels[TARGETS].notna().any(axis=1).sum()} / {len(train)}")

In [ ]:
# How good is the text-only labeller? Score it against the 58 gold studies.
# "Unknown" is scored as 0.5 -- i.e. an uninformative guess -- so this number is a fair
# lower bound on what the reports alone give you.
from sklearn.metrics import roc_auc_score

gold_weak = weak_labels.loc[gold_mask.values, TARGETS].reset_index(drop=True)
y_gold = gold[TARGETS].values.astype(int)

rows, aucs = [], []
for j, name in enumerate(TARGETS):
    p = gold_weak[name].fillna(0.5).values
    auc = roc_auc_score(y_gold[:, j], p) if len(np.unique(y_gold[:, j])) > 1 else np.nan
    aucs.append(auc)
    rows.append((name, int(y_gold[:, j].sum()), gold_weak[name].notna().mean(), auc))

print(pd.DataFrame(rows, columns=["finding", "gold positives", "report coverage", "AUC"])
        .round(3).to_string(index=False))
print(f"\nMACRO AUC of the text rules alone: {np.nanmean(aucs):.4f}")
print("\nRead that as: the reports carry real signal, but the rules are crude -- especially for the")
print("OA columns, where 'coverage' is low. Improving this cell is the highest-leverage work in")
print("the whole notebook, because every image model downstream trains on these labels.")

## 4. From DICOM to a fixed-size tensor

A study is a bag of series of different lengths, resolutions, planes and pulse sequences. A CNN
wants a fixed-shape tensor. The reduction used here — and, in a larger form, by every strong
solution — is:

1. **Slot selection.** Take one series per (plane, fluid-sensitivity) slot: sagittal-fluid,
   coronal-fluid, axial-fluid. Fall back to any series in that plane if the preferred flag is
   missing. A study with no axial series simply gets a masked, zero-filled slot.
2. **Slice sampling.** Order the series by `InstanceNumber` and sample `N_TRIPLET` positions
   evenly across the middle 15–85% of the stack. The extremes of a knee stack are mostly air and
   skin.
3. **2.5D windows.** At each sampled position take the 3 adjacent slices and stack them as RGB
   channels. This is the cheap way to give a 2D backbone a little volumetric context — and it
   lets you use ImageNet-pretrained weights, which expect 3 channels.
4. **Physical cropping.** Crop `CROP_MM` millimetres around the image centre using
   `PixelSpacing`, *then* resize to `IMG`. Cropping in millimetres rather than pixels makes
   scanners with different fields of view comparable — the knee ends up the same physical size
   in every image.
5. **Intensity normalisation.** MRI has no absolute intensity scale (unlike CT's Hounsfield
   units), so a fixed window is meaningless. Normalise each window by its own 2nd/98th
   percentiles.

The result is `[K, 3, IMG, IMG]` per study plus a `[K]` validity mask. It is cached as `.npz`
so the expensive DICOM decoding happens exactly once.

In [ ]:
import pydicom
import cv2

cv2.setNumThreads(1)  # we parallelise across studies, not inside OpenCV

SERIES_DIR_TRAIN = ROOT / "train_series"
SERIES_DIR_TEST = ROOT / "test_series"
HAVE_IMAGES = SERIES_DIR_TRAIN.exists() or SERIES_DIR_TEST.exists()
print("image directories present:", HAVE_IMAGES)
if not HAVE_IMAGES:
    print("  -> metadata-only environment. The training cells will be skipped and the notebook")
    print("     will still write a valid submission from the report rules.")

series_by_study = {k: v.to_dict("records") for k, v in train_series.groupby("StudyInstanceUID")}
series_by_study_test = {k: v.to_dict("records") for k, v in test_series.groupby("StudyInstanceUID")}


def pick_series(rows, plane, fluid, used):
    """Choose one series for a slot: the requested plane, preferring the requested fluid flag."""
    candidates = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    preferred = [r for r in candidates if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
    if preferred:
        return preferred[0]
    return candidates[0] if candidates else None


def ordered_slices(series_dir: Path):
    """Return [(path, pixel_spacing_mm), ...] ordered along the stack."""
    keyed = []
    for f in glob.glob(str(series_dir / "*.dcm")):
        try:
            hdr = pydicom.dcmread(f, stop_before_pixels=True)
            pos = int(getattr(hdr, "InstanceNumber", 0) or 0)
            spacing = float(hdr.PixelSpacing[0]) if hasattr(hdr, "PixelSpacing") else 0.0
        except Exception:
            continue
        keyed.append((pos, f, spacing))
    keyed.sort()
    return [(f, s) for _, f, s in keyed]


def read_pixels(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr   # MONOCHROME1 stores inverted intensities
        return arr
    except Exception:
        return None


def crop_and_resize(arr, spacing):
    """Keep CROP_MM millimetres around the centre, then resize to IMG x IMG."""
    h, w = arr.shape
    if spacing <= 0:
        spacing = CROP_MM / max(h, w)          # no PixelSpacing tag: assume the FOV is the crop
    side = min(int(round(CROP_MM / spacing)), h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return cv2.resize(arr[y0:y0 + side, x0:x0 + side], (IMG, IMG), interpolation=cv2.INTER_AREA)


def build_study(study_uid: str, rows, series_dir: Path):
    """-> (uint8 [K, 3, IMG, IMG], bool [K]). Missing slots are zeros with mask False."""
    volume = np.zeros((K, 3, IMG, IMG), np.uint8)
    mask = np.zeros(K, bool)
    used, w = set(), 0

    for plane, fluid in SLOTS:
        record = pick_series(rows, plane, fluid, used)
        if record is None:
            w += N_TRIPLET
            continue
        used.add(record["SeriesInstanceUID"])
        files = ordered_slices(series_dir / study_uid / record["SeriesInstanceUID"])
        n = len(files)
        if n == 0:
            w += N_TRIPLET
            continue

        lo, hi = int(n * SLICE_BAND[0]), max(int(n * SLICE_BAND[1]) - 1, 0)
        centres = np.linspace(lo, max(hi, lo), N_TRIPLET).round().astype(int)
        median_spacing = float(np.median([s for _, s in files if s > 0]) if any(s > 0 for _, s in files) else 0.0)

        for centre in centres:
            idx = [int(np.clip(centre + d, 0, n - 1)) for d in (-1, 0, 1)]
            planes, spacings = [], []
            for i in idx:
                path, spacing = files[i]
                planes.append(read_pixels(path))
                spacings.append(spacing if spacing > 0 else median_spacing)
            present = [p for p in planes if p is not None]
            if not present:
                w += 1
                continue
            # One intensity window for the whole triplet, so the 3 channels stay comparable.
            low, high = np.percentile(np.concatenate([p.ravel() for p in present]), [2.0, 98.0])
            for c, (p, spacing) in enumerate(zip(planes, spacings)):
                if p is None:
                    continue
                norm = np.clip((p - low) / (high - low + 1e-6), 0, 1)
                volume[w, c] = (crop_and_resize(norm, spacing) * 255).astype(np.uint8)
            mask[w] = True
            w += 1
    return volume, mask


def cached_study(study_uid: str, rows, series_dir: Path):
    """Build the volume once and reuse it; DICOM decoding is the bottleneck, not the GPU."""
    path = CACHE / f"{study_uid}.npz"
    if path.exists():
        with np.load(path) as z:
            return z["volume"], z["mask"]
    volume, mask = build_study(study_uid, rows, series_dir)
    np.savez_compressed(path, volume=volume, mask=mask)
    return volume, mask

In [ ]:
# Sanity check on one study: decode it and look at the windows.
if HAVE_IMAGES:
    import matplotlib.pyplot as plt

    demo_uid = gold["StudyInstanceUID"].iloc[0]
    t0 = time.time()
    vol, msk = build_study(demo_uid, series_by_study.get(demo_uid, []), SERIES_DIR_TRAIN)
    print(f"{demo_uid[-12:]}  volume {vol.shape}  valid windows {int(msk.sum())}/{K}  "
          f"({time.time() - t0:.1f}s to decode)")
    print("gold labels:", {k: int(v) for k, v in gold.iloc[0][TARGETS].items() if v == 1} or "none positive")

    fig, axes = plt.subplots(N_SLOT, N_TRIPLET, figsize=(3 * N_TRIPLET, 3 * N_SLOT))
    for s, (plane, _) in enumerate(SLOTS):
        for t in range(N_TRIPLET):
            ax = axes[s, t]
            ax.imshow(vol[s * N_TRIPLET + t, 1], cmap="gray")   # channel 1 = the centre slice
            ax.set_title(f"{plane} #{t}", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. The model

Nothing exotic — but note the **head**, because it is the part that matters and it is a
simplified version of what all four branches of the 0.936 ensemble do.

The backbone embeds each of the `K` windows independently into a feature vector. We then need
to collapse `K` vectors into 12 predictions. Mean-pooling would be wrong: a Baker's cyst is
visible in maybe two of the twelve windows, and averaging dilutes it towards nothing.

Instead we use **per-label attention**: for each of the 12 findings the head learns its own
softmax over the `K` windows, so it can pool from wherever *that* finding is actually visible —
the sagittal windows for the ACL, the axial ones for a Baker's cyst. Missing slots are masked
out of the softmax so a study with no axial series is not penalised.

```
[B, K, 3, H, W] --backbone--> [B, K, F] --attention(12 heads)--> [B, 12, F] --dot--> [B, 12]
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
torch.manual_seed(SEED)

# torch >= 2.4 moved the AMP helpers out of torch.cuda.amp; support both.
try:
    from torch.amp import GradScaler as _GradScaler, autocast as _autocast

    def make_scaler():
        return _GradScaler("cuda", enabled=USE_AMP)

    def amp_autocast():
        return _autocast("cuda", enabled=USE_AMP)
except ImportError:                                    # torch < 2.4
    def make_scaler():
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

    def amp_autocast():
        return torch.cuda.amp.autocast(enabled=USE_AMP)

print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class KneeStudyDataset(Dataset):
    """One item = one study: K windows, a validity mask, and the 12 (possibly unknown) labels."""

    def __init__(self, uids, labels, series_map, series_dir, train_mode=False):
        self.uids = list(uids)
        self.labels = np.asarray(labels, np.float32)   # NaN allowed = unknown
        self.series_map = series_map
        self.series_dir = series_dir
        self.train_mode = train_mode

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        uid = self.uids[i]
        volume, mask = cached_study(uid, self.series_map.get(uid, []), self.series_dir)
        x = torch.from_numpy(volume.astype(np.float32) / 255.0)      # [K, 3, H, W]
        if self.train_mode and np.random.rand() < 0.5:
            x = torch.flip(x, dims=[-1])                              # horizontal flip
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        y = torch.from_numpy(self.labels[i])
        target_mask = ~torch.isnan(y)                                 # which labels are usable
        return x, torch.from_numpy(mask.astype(np.float32)), torch.nan_to_num(y), target_mask


class KneeNet(nn.Module):
    def __init__(self, arch=BACKBONE, n_label=N_LABEL, pretrained=PRETRAINED, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0,
                                          in_chans=3, global_pool="avg")
        dim = self.backbone.num_features
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                  nn.Linear(256, n_label))
        self.cls_w = nn.Parameter(torch.zeros(n_label, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_label))
        nn.init.trunc_normal_(self.cls_w, std=0.02)

    def forward(self, x, window_mask):
        b, k = x.shape[:2]
        feats = self.backbone(x.flatten(0, 1)).view(b, k, -1)          # [B, K, F]
        h = self.norm(feats)
        logits_attn = self.attn(h)                                     # [B, K, 12]
        # Zero-filled slots must not attract attention.
        logits_attn = logits_attn.masked_fill(window_mask[:, :, None] < 0.5, float("-inf"))
        # A study with no valid window at all would give an all-NaN softmax; fall back to uniform.
        empty = window_mask.sum(1) == 0
        if empty.any():
            logits_attn[empty] = 0.0
        weights = torch.softmax(logits_attn, dim=1)                    # [B, K, 12]
        pooled = torch.einsum("bkn,bkf->bnf", weights, h)              # [B, 12, F]
        return (pooled * self.cls_w).sum(-1) + self.cls_b              # [B, 12]


def masked_bce(logits, targets, target_mask):
    """BCE that ignores labels the report did not tell us about."""
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    loss = loss * target_mask.float()
    denom = target_mask.float().sum().clamp(min=1.0)
    return loss.sum() / denom


def macro_auc(y_true, y_prob):
    scores = [roc_auc_score(y_true[:, j], y_prob[:, j])
              for j in range(y_true.shape[1]) if len(np.unique(y_true[:, j])) > 1]
    return float(np.mean(scores)), scores

## 6. Train

**The split.** The 58 gold studies are the validation set and are *never* trained on. Everything
else — the studies where the rules produced at least one usable label — is training data.

**What to expect.** With 600 studies, `resnet18` and 4 epochs, this is a smoke test, not a
competitive model: expect a gold macro AUC somewhere in the 0.60–0.75 range, bouncing around by
several points between runs. With 58 validation studies and as few as 9 positives in some
columns, a ±0.05 swing is noise. Do not chase it. Raise `MAX_TRAIN_STUDIES` first; it is the
variable that actually moves the score.

In [ ]:
usable = weak_labels[TARGETS].notna().any(axis=1) & ~gold_mask.values
train_uids = weak_labels.loc[usable, "StudyInstanceUID"].tolist()
train_y = weak_labels.loc[usable, TARGETS].values.astype(np.float32)

rng = np.random.default_rng(SEED)
if len(train_uids) > MAX_TRAIN_STUDIES:
    keep = rng.choice(len(train_uids), MAX_TRAIN_STUDIES, replace=False)
    train_uids = [train_uids[i] for i in keep]
    train_y = train_y[keep]

val_uids = gold["StudyInstanceUID"].tolist()
val_y = gold[TARGETS].values.astype(np.float32)

print(f"training on {len(train_uids)} weakly-labelled studies "
      f"({np.isfinite(train_y).mean() * 100:.1f}% of label cells known)")
print(f"validating on {len(val_uids)} gold studies")

In [ ]:
if HAVE_IMAGES:
    train_ds = KneeStudyDataset(train_uids, train_y, series_by_study, SERIES_DIR_TRAIN, train_mode=True)
    val_ds = KneeStudyDataset(val_uids, val_y, series_by_study, SERIES_DIR_TRAIN)
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

    model = KneeNet().to(DEVICE)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    optimiser = torch.optim.AdamW(
        [{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
         {"params": head_params, "lr": LR_HEAD}], weight_decay=1e-4)
    steps = max(EPOCHS * len(train_dl), 1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimiser, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.2)
    scaler = make_scaler()

    def predict(loader):
        model.eval()
        out = []
        with torch.no_grad():
            for x, wm, _, _ in loader:
                x, wm = x.to(DEVICE), wm.to(DEVICE)
                with amp_autocast():
                    out.append(torch.sigmoid(model(x, wm)).float().cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, N_LABEL), np.float32)

    best_auc, best_state = -1.0, None
    for epoch in range(EPOCHS):
        model.train()
        running, seen, t0 = 0.0, 0, time.time()
        for x, wm, y, ym in train_dl:
            x, wm, y, ym = x.to(DEVICE), wm.to(DEVICE), y.to(DEVICE), ym.to(DEVICE)
            optimiser.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = masked_bce(model(x, wm), y, ym)
            scaler.scale(loss).backward()
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()
            running += loss.item() * x.size(0)
            seen += x.size(0)

        val_prob = predict(val_dl)
        auc, per_label = macro_auc(val_y.astype(int), val_prob)
        print(f"epoch {epoch + 1}/{EPOCHS}  loss {running / max(seen, 1):.4f}  "
              f"gold macro AUC {auc:.4f}  ({time.time() - t0:.0f}s)")
        if auc > best_auc:
            best_auc, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(best_state, WORK / "knee_starter.pt")
    print(f"\nbest gold macro AUC: {best_auc:.4f}   (58 studies -- treat as a smoke test, not a score)")
    print(pd.Series(dict(zip(TARGETS, macro_auc(val_y.astype(int), predict(val_dl))[1]))).round(3).to_string())
else:
    model = None
    print("No images available -- skipping training.")

## 7. Predict and submit

Submission format: one row per test `StudyInstanceUID`, 12 probability columns. The visible
`test.csv` has only a handful of studies — the real test set is substituted when the notebook is
rerun, so **your code must handle studies it has never seen and must not depend on the internet
or on wall-clock assumptions**. Wrap per-study inference in `try/except` and fall back to a
neutral 0.5; a crash on one bad study fails the whole submission.

In [ ]:
sub = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"].astype(str)})
for c in TARGETS:
    sub[c] = 0.5

if HAVE_IMAGES and model is not None:
    model.eval()
    test_ds = KneeStudyDataset(sub["StudyInstanceUID"].tolist(),
                               np.full((len(sub), N_LABEL), np.nan, np.float32),
                               series_by_study_test, SERIES_DIR_TEST)
    for i in range(len(test_ds)):
        try:
            x, wm, _, _ = test_ds[i]
            with torch.no_grad(), amp_autocast():
                logits = model(x[None].to(DEVICE), wm[None].to(DEVICE))
            sub.loc[i, TARGETS] = torch.sigmoid(logits)[0].float().cpu().numpy()
        except Exception as e:                       # never let one study sink the submission
            print(f"fallback to 0.5 for study {i}: {e}")

sub.to_csv(WORK / "submission.csv", index=False)
print(sub.head().to_string(index=False))
print("\nwrote", WORK / "submission.csv")

## 8. Where to go from here

Ordered by expected return, from the evidence in this competition:

**1. Better labels — by far the biggest lever.**
The keyword rules in section 3 reach roughly 0.68 macro AUC on the gold set and leave most
`(study, finding)` cells unknown. Replace them with an LLM that reads each report and emits 12
**soft** probabilities. That is what the 0.936 solution did, and its own notes record the payoff:
corpus 3,155 → 4,349 studies moved the gold score 0.8923 → 0.9054, concentrated in the labels
the rules handle worst (Lateral Meniscus +0.071, Fracture +0.057, Lateral OA +0.048). Soft
targets also regularise: training on 0.7 instead of 1.0 stops the model memorising the teacher's
mistakes.

**2. More data and more slices.**
Raise `MAX_TRAIN_STUDIES` to everything usable, `N_TRIPLET` to 8–16, `IMG` to 336. Pre-cache the
volumes in a separate Kaggle dataset so training runs do not re-decode DICOM.

**3. More slots.**
The strong solutions use six slots, not three: sagittal/coronal/axial × fluid-sensitive and
structural (T1/PD without fat suppression). T1 is where you read cartilage and bone, which is
exactly where `Medial OA` / `Lateral OA` live — the two hardest columns.

**4. A stronger backbone.**
`resnet18` → `convnext_tiny`, `coatnet_rmlp_2_rw_384`, or a DINOv2/v3 ViT. The 0.936 notebook's
own ablation on 45 gold studies: CoAtNet 384 → 0.9025, SwinBase 384 → 0.8825, EffNetV2-L 480 →
0.8716. Medical pre-training (RadImageNet) is worth trying too — it is one of the four branches
precisely because it makes different mistakes.

**5. Validate better than the 58 gold studies allow.**
Hold out part of your weakly-labelled corpus and evaluate against the report-derived labels as
well. It is a noisier target but has hundreds of studies behind it, so a 0.005 difference there
means more than a 0.03 difference on 58 studies.

**6. Ensemble last, not first.**
Rank-average several folds and architectures. But note the ablation in `rsna-baseline.ipynb`:
blending three architectures bought +0.001 on the public leaderboard over a single CoAtNet, at
3× the inference cost. Ensembling is the final polish; it will not rescue weak labels.

**7. Watch the runtime budget.**
This is a rerun competition with a hard time limit and no internet. Cache aggressively, decode
DICOM headers before pixels, thread the I/O, and always keep a fallback path that writes a valid
`submission.csv` even when something fails.